### Process foraging data

In [ ]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

# Path setup
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path().absolute().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"
AMPLIO_DIR = PROJECT_ROOT / "amplio"
RESULTS_DIR = PROJECT_ROOT / "results" / "figures"

print(f"Project root: {PROJECT_ROOT}")

### Add creativity scores to dataframe:
Problem statement: 2 needed columns are in one CSV coded with P-IDs. Need to get these values only for those in final data (n = 464), so need to cross-reference from another csv first.

In [ ]:
final_pids = pd.read_csv(PROCESSED_DIR / 'income_newgroup.csv')
pid = final_pids['PID_new'].tolist()
subj_id = final_pids['Subj'].tolist()
print(len(subj_id))
df_with_creativity = pd.read_csv(PROCESSED_DIR / 'alt_creativity_vals.csv')
df = df_with_creativity[df_with_creativity['PID_new'].isin(pid)]
df['Subj'] = subj_id

df.to_csv(PROCESSED_DIR / '464withcreativity.csv')

### Add income to final dataframe:

In [ ]:
original_dataframe = pd.read_csv(PROCESSED_DIR / 'total_data_vludwig.csv', encoding='UTF-8')
current_dataframe = pd.read_csv(PROCESSED_DIR / 'fADHD_final_groupDF.csv')
orig_users = original_dataframe['Subj'].tolist()
orig_users_income = original_dataframe['Income'].tolist()
current_users = current_dataframe['ID'].tolist()

print(len(current_users))
print(len(orig_users))

# Keep only the users that are in the current dataframe:
newgroup = list(set(orig_users) & set(current_users))
print(len(newgroup))
df = original_dataframe[original_dataframe['Subj'].isin(newgroup)]
income = df['Income'].tolist()
df.to_csv(PROCESSED_DIR / 'income_newgroup.csv')


In [ ]:
ethnic_data = pd.read_csv(PROCESSED_DIR / 'ethnicdata.csv')
ethnic_ids = list(set(ethnic_data['ID']))
print(len(ethnic_ids))
data_ids = list(set(data['ID']))
print(len(data_ids))

# Remove different IDs from ethnic_data dataframe:
newgroup = list(set(ethnic_ids)-set(data_ids))
print(len(newgroup))
clean_ethnic = ethnic_data[~ethnic_data['ID'].isin(newgroup)]
print(len(clean_ethnic))
clean_ethnic.to_csv(PROCESSED_DIR / 'cleanethnic.csv', index=False)

### Amplio Pre-processing

Steps:
1. Clean columns [Excel]
2. Split event col into Block and BushRound col [Excel]
3. Add 1/0 decision col and change time to seconds [Py]
4. Drop 'block' from col and add commas to BushRound col [Py]
5. Separate BushRound col into Bush and Round cols [Excel]
6. Drop 'bush' and 'round' text from cols [Py]
7. Sort data: ID --> Block --> Bush --> Round [Excel]
8. Remove (23) individuals who contain no "1" decisions [Py]
9. Remove final bushes of all users [Py]
10. Remove individuals with less than 25 decisions [Py]

In [ ]:
''' Step 3: convert Decisions to 0/1 and TotalTime to seconds.'''

# Load data:
amp  = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_2.csv')
decs = amp['Decision'].tolist()

# Code 0/1 decisions:
decs = [0 if element == 'explore' else element for element in decs]
decs = [1 if element == 'exploit' else element for element in decs]
amp['Decision'] = decs

# Convert TotalTime as seconds
ms_time = amp['TotalTime_ms'].tolist()
ms_time = [element/1000 for element in ms_time]
amp['TotalTime_sec'] = ms_time

# # Drop unnamed columns and save:
# amp.head()

amp = amp.drop(['TotalTime_ms'], axis=1)
amp.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_3.csv')

In [ ]:
''' Step 4: separate event data into block, bush, and round cols.'''

# Load user list (241 unique) data:
amp = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_3.csv')

# Remove 'block' and keep number:
event = amp['Event'].tolist()
event = [element[5:] for element in event] # drop first 5 elements of each row
amp['Event'] = event # save back to df

# Add commas to bushround col:
bush = amp['Bush'].tolist()
bush_clean = []

# Add a comma after the first 5 elements:
for element in bush:
    if element[5] != 'r':
        element = element[:6] + ',' + element[6:]
        bush_clean.append(element)
    else:
        element = element[:5] + ',' + element[5:]
        bush_clean.append(element)

# Add to dataframe amp and save:
amp['Bush'] = bush_clean
amp.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_4.csv')

In [ ]:
''' Step 6: further clean bush and round data.'''
amp = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_5.csv')
bush = amp['Bush'].tolist()
rounds = amp['Round'].tolist()

bush = [element[4:] for element in bush]
amp['Bush'] = bush
rounds = [element[5:] for element in rounds]
amp['Round'] = rounds

amp.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_6.csv')

In [ ]:
'''
Step 8: Remove individual if no 1 decisions.
'''
amp = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_7.csv')
users = sorted(set(amp['ID'].tolist()))

print(f'There are {len(users)} users.')
users_with_no_exploits = []
for i in range(len(users)):
    user = users[i]
    user = amp[amp['ID'] == user]
    userblock1 = user[user['Block'] == 1]
    userblock2 = user[user['Block'] == 2]
    userblock1decision = userblock1['Decision'].tolist()
    userblock2decision = userblock2['Decision'].tolist()
    if userblock1decision.count(1) == 0 or userblock2decision.count(1) == 0:
        users_with_no_exploits.append(users[i])

print(f'There are {len(users_with_no_exploits)} users that made no exploit decisions.')

# Remove users with no exploits:

# 2 lists: users and users_with_no_exploits
clean1 = amp[~amp['ID'].isin(users_with_no_exploits)]
clean1users = sorted(set(clean1['ID'].tolist()))
print(f'There are {len(clean1users)} users after removing users with no exploits.')
clean1.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_8.csv')

In [ ]:
'''
Step 9: Remove final bush of all individuals.
'''
amp = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_8.csv')
users = sorted(set(amp['ID'].tolist()))
print(f'There are {len(users)} users.')

# Remove final bush of all users:
amp2 = pd.DataFrame([])

for i in range(len(users)):
    user = users[i]
    df = amp[amp['ID'] == user]
    
    df1 = df[df['Block']==1]
    bush1 = df1['Bush'].tolist()
    bush1 = bush1[-1]
    df1 = df1[df1['Bush'] != bush1]

    df2 = df[df['Block']==2]
    bush2 = df2['Bush'].tolist()
    bush2 = bush2[-1]
    df2 = df2[df2['Bush'] != bush2]

    # # Save to new dataframe:
    amp2 = amp2.append(df1)
    amp2 = amp2.append(df2)
    amp2.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_9.csv')

In [ ]:
'''
Step 10: Remove individuals with less than 25 decisions.
'''
amp = pd.read_csv(PROCESSED_DIR / 'AMPLIO_Mar622_9.csv')
users = sorted(set(amp['ID'].tolist()))

print(f'There are {len(users)} users.')
users_withlessthan25 = []
for i in range(len(users)):
    user = users[i]
    user = amp[amp['ID'] == user]
    # userblock1 = user[user['Block'] == 1]
    # userblock2 = user[user['Block'] == 2]
    userdecisions = user['Decision'].tolist()
    # userblock2decision = userblock2['Decision'].tolist()
    if len(userdecisions) < 25:
        users_withlessthan25.append(users[i])

print(f'There are {len(users_withlessthan25)} users with less than 25 decisions.')

# Remove users with less than 25 decisions:
clean1 = amp[~amp['ID'].isin(users_withlessthan25)]
clean1users = sorted(set(clean1['ID'].tolist()))
print(f'There are {len(clean1users)} users after removing users with less than 25 decisions.')
clean1.to_csv(PROCESSED_DIR / 'AMPLIO_Mar622_10.csv')

### other

In [ ]:
ppl = pd.read_csv(PROCESSED_DIR / 'fADHD_groupsum_022122.csv')

adhd_ct = ppl['ADHD Diagnosis'].tolist()

# Convert nan to 0
adhd_ct = [0 if x == 'NA' else x for x in adhd_ct]
# print(adhd_ct)
print(f'ADHD count: {adhd_ct.count(1)}') # 25
print(f'Non-ADHD count: {adhd_ct.count(0)}') # 434
# Separate ppl based on adhd_diagnosis:
ppl_adhd = ppl[ppl['ADHD Diagnosis'] == 1]
ppl_nonadhd = ppl[ppl['ADHD Diagnosis'] == 0]
asrs_adhd = ppl_adhd['ASRS-A Score'].tolist()
asrs_nonadhd = ppl_nonadhd['ASRS-A Score'].tolist()
print(f'Number of ppl with ADHD: {len(ppl_adhd)}')
print(f'Number of ppl without ADHD: {len(ppl_nonadhd)}')
# Print averages
print(f'Avg asrs of ppl with ADHD: {np.mean(asrs_adhd)}')
print(f'Avg asrs of ppl without ADHD: {np.mean(asrs_nonadhd)}')
# Print standard deviations
print(f'Std asrs of ppl with ADHD: {np.std(asrs_adhd)}')
print(f'Std asrs of ppl without ADHD: {np.std(asrs_nonadhd)}')

In [ ]:
# Plot average asrs for asrs_adhd and asrs_nonadhd:
# Avg asrs of ppl with ADHD: 12.08
# Avg asrs of ppl without ADHD: 7.921658986175115
# Std asrs of ppl with ADHD: 3.4631777315061383
# Std asrs of ppl without ADHD: 4.0290810393883145

fig = plt.figure(figsize=(10,10))
plt.bar([1,2], [np.mean(asrs_adhd), np.mean(asrs_nonadhd)], yerr=[np.std(asrs_adhd), np.std(asrs_nonadhd)], capsize=10, color=['gray', 'gray'])
plt.xticks([1,2], ['ADHD', 'Non-ADHD'])
plt.ylabel('ASRS-A Score')
plt.title('Average ASRS-A Score by Diagnosis')
plt.savefig(PROCESSED_DIR / 'asrs_by_diagnosis.svg')
plt.show()

In [ ]:
# Check participants that we excluded:
total_ppl = pd.read_csv(PROCESSED_DIR / 'total_data_vludwig.csv')
curr_ppl = pd.read_csv(PROCESSED_DIR / 'fADHD_groupsum_021522.csv')
total_ids = total_ppl['Subj'].tolist()
curr_ids = curr_ppl['ID'].tolist()

excluded_ids = list(set(total_ids)-set(curr_ids))
# print(len(excluded_ids))
excluded_ppl = total_ppl[total_ppl['Subj'].isin(excluded_ids)]
excluded_ppl.to_csv(PROCESSED_DIR / 'excluded_ppl.csv', index=False)

In [ ]:
# import foraging group data
data = pd.read_csv(PROCESSED_DIR / 'fADHD_groupsum_021522.csv')
# sxs = data['Gender'].tolist()
# ages = data['Age'].tolist()
# data.head()

# count how many 0's are in sxs:
# print(sxs.count())
# 221/464 # % of females is 47.63%
# # print(sum(ages)/len(ages)) # average age is 45.72 

# import statistics
# # get

# stddev_ages= statistics.pstdev(ages)
# print(stddev_ages)
ethnlist = data['Ethnic'].tolist()

# turn nan to 7:
ethnlist = [7 if x == np.nan else x for x in ethnlist]
# Count the ethnicities in ethnclist:
print('There are ' + str(ethnlist.count(1)) + ' white caucasian (non-Hispanic) folks.')
print('There are ' + str(ethnlist.count(2)) + ' AA or black folks.')
print('There are ' + str(ethnlist.count(3)) + ' span/hisp/latino folks.')
print('There are ' + str(ethnlist.count(4)) + ' asian folks.')
print('There are ' + str(ethnlist.count(5)) + ' samerindian folks.')
print('There are ' + str(ethnlist.count(6)) + ' middle eastern folks.')
print('There are ' + str(ethnlist.count(7)) + ' unknown folks.') # not working but 7 left over


In [ ]:
# import allsurveys.csv
surveys = pd.read_csv(PROCESSED_DIR / 'allsurveys.csv')
# surveys.head()

id_from_trials = list(set(surveys['Subj']))
print(len(id_from_trials))

path2currentdf = PROCESSED_DIR / 'fADHD_final_groupDF.csv'
currentdf = pd.read_csv(path2currentdf)
id_from_groups = list(set(currentdf['ID']))
print(len(id_from_groups))

# Subtract the list difference:
id_from_trials_minus_groups = list(set(id_from_trials) - set(id_from_groups))

# Remove diff from newsurveys dataframe:
newdf = surveys[~surveys['Subj'].isin(id_from_trials_minus_groups)]
print(len(newdf))
# Convert newsurveys to csv:
newdf.to_csv(PROCESSED_DIR / 'newsurveys.csv', index=False)

# save finsurveys to csv


In [ ]:
# Import data:
df_trials = pd.read_csv(PROCESSED_DIR / 'trials_final.csv')
# df_trials.head()

path2currentdf = PROCESSED_DIR / 'fADHD_final_groupDF.csv'
currentdf = pd.read_csv(path2currentdf)
id_from_groups = list(set(currentdf['ID']))
print(len(id_from_groups))

id_from_trials = list(set(df_trials['ID']))
print(len(id_from_trials))

# Compare the two lists:
id_from_groups_not_in_trials = [x for x in id_from_groups if x not in id_from_trials]
print(len(id_from_groups_not_in_trials))

In [ ]:
# Import the dataset:
group = pd.read_csv('group.csv')
group['ADHD_diagnosis'].replace({'no': 0, 'yes': 1}, inplace=True)
trials = pd.read_csv('trials_final.csv')
trials['BerryCount'] = trials['BerryCount'].fillna(0)

# Show the first 5 rows of data:
# trials.head()

dev = pd.read_csv('devdata.csv')
trialsID = list(set(trials['ID'].tolist()))
# Kep only the trials that are in the trials dataset:
groupDF = dev[dev['ID'].isin(trialsID)]
# print columns of the groupDF:
groupDF.columns
# save groupDF to csv:
print(len(groupDF))
# groupDF.to_csv('groupDF.csv', index=False)

In [ ]:
path2newsurveys = PROCESSED_DIR / 'addtlsurveys.csv'
newsurveys = pd.read_csv(path2newsurveys)
newsurveysID = list(set(newsurveys['ID'].tolist()))

# print(len(newsurveysID))

diff = list(set(newsurveysID) - set(currdfID)) # get difference

# Remove diff from newsurveys dataframe:
newsurveys = newsurveys[~newsurveys['ID'].isin(diff)]
# Convert newsurveys to csv:
newsurveys.to_csv('newsurveys.csv', index=False)

# print(newsurveys)

# # Convert MW_task column to float:
# newsurveys['MW_task'] = newsurveys['MW_task'].astype(float)
# # Convert Life_Sat column to float:
# newsurveys['Life_Sat'] = newsurveys['Life_Sat'].astype(float)
# # Convert MW_S_check_correct column to float:
# newsurveys['MW_S_check_correct'] = newsurveys['MW_S_check_correct'].astype(float)
# # Convert Perma_Pos column to float:
# newsurveys['Perma_Pos'] = newsurveys['Perma_Pos'].astype(float)

# # Move the new surveys to the current dataframe:
# currentdf = currentdf.append(newsurveys, ignore_index=True)
# currentdf.head()

In [ ]:
# get user id of individuals with less than 25 trials:
trials1 = pd.read_csv('devdata.csv')
print(len(trials1))
print('User id of individuals with less than 25 trials:')
lessthan25 = trials1[trials1['NumOfDecisions'] < 25]['ID']
print(lessthan25)

In [ ]:

# o White Caucasian – Non Hispanic (1) ; # o African-American or Black (2); # o Spanish, Hispanic, or Latino (3) ; # o Asian (4) ; # o American Indian, Alaskan Native (5)
# o Middle Eastern (6) ; # o Unknown, other, or not reported (7)

# 1. add ethnicity data from another csv and cross-reference the IDs to only include those IDs in trials.csv:
# 2. bar plot of adhd vs asrs-a score w error bars
ethnic = pd.read_csv('ethnicdata.csv')
ethnic = ethnic[ethnic['ID'].isin(trialsID)]
ethnlist = ethnic['Ethnic'].tolist()

# turn nan to 7:
ethnlist = [7 if x == np.nan else x for x in ethnlist]
# Count the ethnicities in ethnclist:
print('There are ' + str(ethnlist.count(1)) + ' white caucasian (non-Hispanic) folks.')
print('There are ' + str(ethnlist.count(2)) + ' AA or black folks.')
print('There are ' + str(ethnlist.count(3)) + ' span/hisp/latino folks.')
print('There are ' + str(ethnlist.count(4)) + ' asian folks.')
print('There are ' + str(ethnlist.count(5)) + ' samerindian folks.')
print('There are ' + str(ethnlist.count(6)) + ' middle eastern folks.')
print('There are ' + str(ethnlist.count(7)) + ' unknown folks.') # not working but 7 left over
# sum of all
# # count NaNs in ethnlist:
# print('There are ' + str(ethnlist.count(np.nan)) + ' missing')

# 

In [ ]:
# Extract the gender and age column from the dataset:

# dev['Gender'] = dev['Gender'].replace(np.nan,4)
gender = groupDF['Gender'].tolist()
adhddiagnosis = groupDF['ADHD Diagnosis'].tolist()
age = groupDF['Age'].tolist()
adhdscore = groupDF['ASRS-A Score'].tolist()
traitanx = groupDF['Trait Anxiety'].tolist()
stateanx = groupDF['State Anxiety'].tolist()

# Count items in list:
print('Female count is: ' + str(gender.count(0)) + ' | Male count is: ' + str(gender.count(1)))
print('Other count is: ' + str(gender.count(2)) + ' | NA count is: ' + str(len(gender)-gender.count(2)-gender.count(1)-gender.count(0)))
print('Male %: ', round((gender.count(0)/len(gender))*100))
print('Average age is: ' + str(np.mean(age)) + '| SD = ' + str(np.std(age)))

sns.countplot(x='Gender', data=groupDF, palette="mako_r")
plt.xlabel("Sex (0 = female, 1= male)")
plt.show()

In [ ]:
#==============# PARTICIPANT SUMMARY STATISTICS #================#

# Cast user id to list and initialize other lists:
users = list(dict.fromkeys(trials['ID'].tolist()))
groupRTlist, mean_RT_ind, std_RT_ind = [], [] , []
Bct, trialnum = [], []
exploretot, exploittot, explratio = [], [], []

for i in range(len(users)):
    # Skip user if not in the group:
    if users[i] not in group['ID'].tolist():
        continue
    # Get the user's reaction times and berry counts:
    userRT = trials[trials['user_id'] == i]['RT'].tolist()
    userBC = trials[trials['user_id'] == i]['BerryCount'].tolist()
    # Get the mean reaction time, std RT, and sum of berry count for that user:
    mean_RT_ind.append(round(np.mean(userRT),3))
    std_RT_ind.append(round(np.std(userRT),3))
    Bct.append(sum(userBC))
    trialnum.append(len(userRT))
    exploretot.append(userBC.count(0))
    exploittot.append((len(userRT) - userBC.count(0)))
    # why division by zero?
    # explratio.append(round(userBC.count(0)/len(userRT),3))
    # explratio.append(round((len(userRT) - userBC.count(0))/userBC.count(0),3))
    # Append the user id to the list:
    groupRTlist.append(i)

# Get game data for each user:

#### FAST CONDITION ####
fast = trials[trials['traveltime'] == 1]
fasttip = fast[fast['Decision'] == 0]
mean_fRTind, std_fRTind = [], []
fBct, ftrialnum, ftipavgs = [], [], []
exploreftot, exploitftot = [], []

for i in range(len(users)):
    
    # Get the user's reaction times and berry counts:
    userfRT = fast[fast['user_id'] == i]['RT'].tolist()
    userfBC = fast[fast['user_id'] == i]['BerryCount'].tolist()
    userftip = fasttip[fasttip['user_id'] == i]['TotalTime_sec'].tolist()
    # Get the mean reaction time, std RT, and sum of berry count for that user:
    mean_fRTind.append(round(np.mean(userfRT),3))
    std_fRTind.append(round(np.std(userfRT),3))
    fBct.append(sum(userfBC))
    ftrialnum.append(len(userfRT))
    exploreftot.append(userfBC.count(0))
    exploitftot.append((len(userfRT) - userfBC.count(0)))
    ftipavgs.append(round(np.mean(userftip),3))

#### SLOW CONDITION ####
slow = trials[trials['traveltime'] == 5]
slowtip = slow[slow['BerryCount'] == 0]
mean_sRTind, std_sRTind = [], []
sBct, strialnum, stipavgs = [], [], []
explorestot, exploitstot = [],[]
# Get the user's reaction times:
for i in range(len(users)):
    # Get the user's reaction times and berry counts:
    usersRT = slow[slow['user_id'] == i]['RT'].tolist()
    usersBC = slow[slow['user_id'] == i]['BerryCount'].tolist()
    usersTip = slowtip[slowtip['user_id'] == i]['TotalTime_sec'].tolist()
    # Get the mean reaction time, std RT, and sum of berry count for that user:
    mean_sRTind.append(round(np.mean(usersRT),3))
    std_sRTind.append(round(np.std(usersRT),3))
    sBct.append(sum(usersBC))
    strialnum.append(len(usersRT))
    explorestot.append(usersBC.count(0))
    exploitstot.append((len(usersRT) - usersBC.count(0)))
    stipavgs.append(round(np.mean(usersTip),3))

### Create dataframe with new variables

In [ ]:
# Create a dataframe with relevant variables:

# Remove decimal from gender and age columns (AKA round them to int):
age = [int(x) for x in age]
gender = [int(x) for x in gender]

# Note: 'decisions' = trials
# Missing exploitexploreratio
devdata = pd.DataFrame({'user_id': groupRTlist, 'ID': users, 'Avg RT (ms)': mean_RT_ind, 'Std RT': std_RT_ind, 
    'TotalReward': Bct, 'NumOfDecisions': trialnum, 'NumOfExploreDecisions (i.e., # of patches)': exploretot, 'NumOfExploitDecisions': exploittot, 
     'Gender': gender, 'Age': age, 'ADHD Diagnosis': adhddiagnosis, 'ASRS-A Score':adhdscore, 
    'Trait Anxiety': traitanx, 'State Anxiety':stateanx, 'Avg Time in Fast Patch (s)': ftipavgs, 'Avg RT in Fast (ms)':mean_fRTind, 'Std RT in Fast': std_fRTind, 
    'Total Reward in Fast': fBct, 'NumOfDecs in Fast': ftrialnum, 'NumofExploits in Fast': exploitftot, 'NumOfPatches in Fast': (exploreftot), 'Avg Time in Slow Patch (s)': stipavgs, 'Avg RT in Slow (ms)':mean_sRTind, 'Std RT in Slow': std_sRTind,
    'Total Reward in Slow': sBct, 'NumOfDecs in Slow': strialnum, 'NumOfExploits in Slow': exploitstot, 'NumOfPatches in Slow':(explorestot)})

# Check with arrays aren't of the same length:
# print(len(gender), len(age), len(adhddiagnosis), len(adhdscore), len(traitanx), len(stateanx))
devdata.head()

# Save devdata to csv:
# devdata.to_csv('./devdata.csv',index=False)


In [ ]:
# Get mean and std age:

userage = devdata['Age'].tolist()
avgage = round(np.mean(userage),3)
stdage = np.std(userage)
print('Average age:', avgage, 'Std age:', stdage)

In [ ]:
# Count how many people reported yes for ADHD diagnosis:
print(devdata['ADHD Diagnosis'].tolist().count(1), 'foragers have an ADHD diagnosis.') # 26
print(len(devdata[devdata['ASRS-A Score']>=17]['ASRS-A Score'].tolist()), 'foragers are likely to have ADHD.') # 16

# Print number of users that either have an ADHD diagnosis or are likely to have it:
adhd_persons = list(set(devdata[devdata['ADHD Diagnosis'] == 1]['user_id'].tolist() + 
            devdata[devdata['ASRS-A Score']>=17]['user_id'].tolist()))
print(str(round((len(adhd_persons)/478),2)*100) + '% ['+ str(len(adhd_persons))+ '] of foragers are either diagnosed or are likely to have ADHD.')

# Print number of users that do not have ADHD nor are likely to have it:
nonadhd_persons = devdata['user_id'].tolist()
for i in range(len(adhd_persons)): nonadhd_persons.remove(adhd_persons[i])
print(str(round((len(nonadhd_persons)/478),2)*100) + '% ['+ str(len(nonadhd_persons))+ '] of foragers are not diagnosed or are unlikely to have ADHD.')

# Add a target column to the dataframe with a 1 if the user is diagnosed with or is likely to have ADHD, 0 if not:
devdata['Target'] = 0
for i in range(len(adhd_persons)): devdata.loc[devdata['user_id'] == adhd_persons[i], 'Target'] = 1

# Replace nan with zero in ADHD diagnosis column:
devdata['ADHD Diagnosis'] = devdata['ADHD Diagnosis'].fillna(0)

devdata['Avg Time in Fast Patch (s)'] = [round(x,3) for x in devdata['Avg Time in Fast Patch (s)']]
devdata['Avg RT in Fast (ms)'] = [round(x,3) for x in devdata['Avg RT in Fast (ms)']]
devdata['Avg RT in Slow (ms)'] = [round(x,3) for x in devdata['Avg RT in Slow (ms)']]
devdata['Avg Time in Slow Patch (s)'] = [round(x,3) for x in devdata['Avg Time in Slow Patch (s)']]
devdata['Std RT in Fast'] = [round(x,3) for x in devdata['Std RT in Fast']]
devdata['Std RT in Slow'] = [round(x,3) for x in devdata['Std RT in Slow']]

# Replace nan with mean in Avg Time in Fast Patch (s) column:
devdata['Avg Time in Fast Patch (s)'] = devdata['Avg Time in Fast Patch (s)'].fillna(devdata['Avg Time in Fast Patch (s)'].mean())
# Replace nan with mean in Avg RT in Fast (ms) column:
devdata['Avg RT in Fast (ms)'] = devdata['Avg RT in Fast (ms)'].fillna(devdata['Avg RT in Fast (ms)'].mean())
# Replace nan with mean in Std RT in Fast (ms) column:
devdata['Std RT in Fast'] = devdata['Std RT in Fast'].fillna(devdata['Std RT in Fast'].mean())
# Replace nan with mean in Avg Time in Slow Patch (s) column:
devdata['Avg Time in Slow Patch (s)'] = devdata['Avg Time in Slow Patch (s)'].fillna(devdata['Avg Time in Slow Patch (s)'].mean())
# Replace nan with mean in Avg RT in Slow (ms) column:
devdata['Avg RT in Slow (ms)'] = devdata['Avg RT in Slow (ms)'].fillna(devdata['Avg RT in Slow (ms)'].mean())
# Replace nan with mean in Std RT in Slow (ms) column:
devdata['Std RT in Slow'] = devdata['Std RT in Slow'].fillna(devdata['Std RT in Slow'].mean())


# devdata.describe() # Check for missing values but also good for population statistics
# devdata.info()


### Visualize data

In [ ]:
# Plot avg RT for nonadhd_persons vs. adhd_persons:
plt.figure(figsize=(10,5))
plt.hist(devdata[devdata['Target'] == 0]['Avg RT (ms)'].tolist(), bins=20, color='blue', label='Non-ADHD')
plt.hist(devdata[devdata['Target'] == 1]['Avg RT (ms)'].tolist(), bins=20, color='red', label='ADHD')
plt.title('Average Reaction Time for Non-ADHD vs. ADHD')
plt.xlabel('Reaction Time (ms)')
plt.ylabel('Number of Users')
plt.legend()
plt.show()

In [ ]:
pd.crosstab(devdata.Age,devdata.Target).plot(kind="bar",figsize=(20,6))
plt.title('ADHD Frequency for Ages')
plt.xlabel('Age')
plt.ylabel('Frequency')
# plt.savefig('adhdagefrq.png')
plt.show()

In [ ]:
# ADHD Frequency for sex
pd.crosstab(devdata.Gender,devdata.Target).plot(kind="bar",figsize=(15,6),color=['#1CA53B','#AA1111' ])
plt.title('ADHD Frequency for Gender')
plt.xlabel('Gender (0 = Female, 1 = Male)')
plt.xticks(rotation=0)
plt.legend(["Unlikely", "Likely"])
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plot total reward against age for each group:
plt.scatter(x=devdata.Age[devdata.Target==1], y=devdata.TotalReward[(devdata.Target==1)], c="red")
plt.scatter(x=devdata.Age[devdata.Target==0], y=devdata.TotalReward[(devdata.Target==0)])
plt.legend(["ADHD", "Not ADHD"])
plt.xlabel("Age")
plt.ylabel("Total Reward")
plt.show()

In [ ]:
# Plot exploreexploitratio against age for each group:
plt.scatter(x=devdata.Age[devdata.Target==1], y=devdata.ExploitExploreRatio[(devdata.Target==1)], c="red")
plt.scatter(x=devdata.Age[devdata.Target==0], y=devdata.ExploitExploreRatio[(devdata.Target==0)])
plt.legend(["ADHD", "Not ADHD"])
plt.xlabel("Age")
plt.ylabel("Explore Exploit Ratio")
plt.show()

In [ ]:
# Plot avg RT for each user in fast condition:
plt.figure(figsize=(10,10))
plt.scatter(devdata['user_id'], devdata['Avg RT in Fast (ms)'], s=10, c='blue')
# plot avg RT for each user in slow condition:
plt.scatter(devdata['user_id'], devdata['Avg RT in Slow (ms)'], s=10, c='red')
plt.title('Average Reaction Time Across Environments')
plt.plot([0,len(users)],[np.mean(devdata['Avg RT in Fast (ms)'])]*2, '--', c='black')
plt.plot([0,len(users)],[np.mean(devdata['Avg RT in Slow (ms)'])]*2, '--', c='green')
plt.plot([0,len(users)],[np.mean(devdata['Avg RT (ms)'])]*2, '--', c='orange')
plt.xlabel('User ID')
plt.ylabel('Average Reaction Time (ms)')
plt.legend(['Fast Avg', 'Slow Avg', 'Global Avg', 'Fast Env', 'Slow Env'])
plt.savefig('fig1.png')
plt.show()

In [ ]:
# Plot spread of patch times against ASRS-A score:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['Avg Time in Fast Patch (s)'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['Avg Time in Slow Patch (s)'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
ax.set_ylabel('Avg Time in Patch (s)')
ax.set_title('ASRS-A Score vs. Avg Time Across Envs')
plt.legend(['Fast Patch', 'Slow Patch'])
plt.savefig('fig2.png')
plt.show()


In [ ]:
#  Plot the number of trials by ASRS-A score:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['NumOfDecisions'], s=50, c='green', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['NumOfDecs in Fast'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['NumOfDecs in Slow'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
ax.set_ylabel('Num of Trials')
plt.legend(['Global', 'Fast', 'Slow'])
ax.set_title('ASRS-A Score vs. Num of Trials')
plt.savefig('fig3.png')
plt.show()


In [ ]:
# Plot total reward in slow vs fast:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['Total Reward in Fast'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['Total Reward in Slow'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
ax.set_ylabel('Total Reward')
plt.legend(['Fast Reward', 'Slow Reward'])
ax.set_title('ASRS-A Score vs. Total Reward')
plt.savefig('fig4.png')
plt.show()

In [ ]:
# Plot total reward vs avg time in patch:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['Avg Time in Fast Patch (s)'], devdata['Total Reward in Fast'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['Avg Time in Slow Patch (s)'], devdata['Total Reward in Slow'], s=50, c='red', alpha=0.25)
ax.set_xlabel('Avg Time in Patch (s)')
ax.set_ylabel('Total Reward')
plt.legend(['Fast Patch', 'Slow Patch'])
ax.set_title('Avg Time in Patch vs. Total Reward')
plt.savefig('fig5.png')
plt.show()



In [ ]:
# Plot total reward vs num of trials:
fig, ax = plt.subplots(figsize=(10,10))
# ax.scatter(devdata['NumOfTrials'], devdata['Total Reward'], s=50, c='green', alpha=0.25)
ax.scatter(devdata['NumOfDecs in Fast'], devdata['Total Reward in Fast'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['NumOfDecs in Slow'], devdata['Total Reward in Slow'], s=50, c='red', alpha=0.25)
ax.set_xlabel('Num of Trials')
ax.set_ylabel('Total Reward')
plt.legend(['Fast', 'Slow'])
ax.set_title('Num of Trials vs. Total Reward')
plt.savefig('fig6.png')
plt.show()

In [ ]:

# Plot total reward vs. avg RT in fast and slow:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['TotalReward'], s=50, c='black', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT (ms)'], s=50, c='green', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Fast (ms)'], s=50, c='blue', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Slow (ms)'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
plt.legend(['Total Reward', 'Avg RT'])
ax.set_ylim([0,2500])
ax.set_ylabel('Total Reward')
ax.set_title('ASRS-A Score vs. Total Reward vs. Avg RT)') 
plt.savefig('fig16.png')
plt.show()

# Plot total reward vs. avg RT in fast and slow:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['TotalReward'], s=50, c='black', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT (ms)'], s=50, c='green', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Fast (ms)'], s=50, c='blue', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Slow (ms)'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
plt.legend(['Total Reward', 'Avg RT in Fast'])
ax.set_ylim([0,2500])
ax.set_ylabel('Total Reward')
ax.set_title('ASRS-A Score vs. Total Reward vs. Avg RT in Fast)') 
plt.savefig('fig17.png')
plt.show()

# Plot total reward vs. avg RT in fast and slow:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['TotalReward'], s=50, c='black', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT (ms)'], s=50, c='green', alpha=0.25)
# ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Fast (ms)'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['Avg RT in Slow (ms)'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_xticks([0,5,10,15,20,25])
plt.legend(['Total Reward', 'Avg RT in Slow'])
ax.set_ylim([0,2500])
ax.set_ylabel('Total Reward')
ax.set_title('ASRS-A Score vs. Total Reward vs. Avg RT in Slow)')
plt.savefig('fig18.png')
plt.show()

In [ ]:
# Plot spread of trait/state anxiety against ADHD diagnosis:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ADHD Diagnosis'], devdata['Trait Anxiety'], s=50, c='blue', alpha=0.25)
# ax.scatter(devdata['ADHD Diagnosis'], devdata['State Anxiety'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ADHD Diagnosis')
ax.set_xticks([0,1])
ax.set_xticklabels(['No', 'Yes'])
ax.set_ylabel('Trait Anxiety')
# plt.legend(['Trait Anxiety', 'State Anxiety'])
ax.set_title('ADHD Diagnosis vs. Trait Anxiety')
plt.savefig('fig10.png')
plt.show()

fig, ax = plt.subplots(figsize=(10,10))
# ax.scatter(devdata['ADHD Diagnosis'], devdata['Trait Anxiety'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ADHD Diagnosis'], devdata['State Anxiety'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ADHD Diagnosis')
ax.set_xticks([0,1])
ax.set_xticklabels(['No', 'Yes'])
ax.set_ylabel('Trait Anxiety')
# plt.legend(['Trait Anxiety', 'State Anxiety'])
ax.set_title('ADHD Diagnosis vs. State Anxiety')
plt.savefig('fig11.png')
plt.show()

## other

In [ ]:
# Plot spread of ADHD score against ADHD diagnosis:

# 1. bar graph
fig, ax = plt.subplots(figsize=(10,10))
# calculate mean and standard deviation for each group:
mean_no = np.mean(devdata['ASRS-A Score'][devdata['ADHD Diagnosis'] == 0])
std_no = np.std(devdata['ASRS-A Score'][devdata['ADHD Diagnosis'] == 0])
mean_yes = np.mean(devdata['ASRS-A Score'][devdata['ADHD Diagnosis'] == 1])
std_yes = np.std(devdata['ASRS-A Score'][devdata['ADHD Diagnosis'] == 1])
# plot mean and standard deviation for each group:
error = [std_no, std_yes]
CTEs = [mean_no, mean_yes]
ax.bar(['No', 'Yes'], CTEs, yerr=error, color=['blue', 'red'], alpha=0.5)
ax.bar(devdata['ADHD Diagnosis'], devdata['ASRS-A Score'])
ax.set_xlabel('ADHD Diagnosis')
ax.set_xticks([0,1])
ax.set_xticklabels(['No', 'Yes'])
ax.set_ylabel('ASRS-A Score')
ax.set_ylim(0,24)
ax.set_yticks([0,5,10,15,20,25])
ax.set_title('ADHD Diagnosis vs. ASRS-A Score')
# plt.savefig('fig12.png')
plt.show()

# 2. print the number of individuals with less than 25 trials:
print('Number of individuals with less than 25 trials:')
print(len(devdata[devdata['NumOfDecisions'] < 25]))

# get user id of individuals with less than 25 trials:
print('User id of individuals with less than 25 trials:')
lessthan25 = devdata[devdata['NumOfDecisions'] < 25]['ID']

# # remove individuals in lessthan25 from devdata and save as new dataframe:
# newdata = devdata[~devdata['ID'].isin(lessthan25)]
# newdata.to_csv('trials_lessthan25.csv')

# # remove individuals with less than 25 trials from trials dataframe:
# trials2 = trials[~trials['ID'].isin(lessthan25)]
# # add userID column that corresponds to subjects in trials2 dataframe:
# trials2subjects = trials['ID'].tolist()
# # for i in range(len(trials2subjects)):
# print(len(list(set(trials2subjects))))
# # trials2.to_csv('trials_notlessthan25.csv')






In [ ]:
# Plot age vs. reward:
plt.figure(figsize=(10,10))
plt.scatter(devdata['Age'],devdata['TotalReward'], s=10)
plt.title('Reward vs. Age')
plt.xlabel('Age')
plt.ylabel('Reward (oz)')
m, b = np.polyfit(devdata['Age'],devdata['TotalReward'], 1)

plt.plot(devdata['Age'], m*devdata['Age'] + b)
plt.savefig('fig13.png')

# Plot gender vs. reward:
plt.figure(figsize=(10,10))
plt.scatter(devdata['user_id'],devdata['TotalReward'], s=10,c=devdata['Gender'])
plt.title('Reward vs. Gender')
plt.xlabel('User ID')
plt.ylabel('Reward (oz)')
plt.legend(['Female', 'Male', 'Other', 'NA'])
plt.savefig('fig14.png')


plt.show()


In [ ]:
# Plot state anxiety vs. asrs-a score:

fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(devdata['ASRS-A Score'], devdata['Trait Anxiety'], s=50, c='blue', alpha=0.25)
ax.scatter(devdata['ASRS-A Score'], devdata['State Anxiety'], s=50, c='red', alpha=0.25)
ax.set_xlabel('ASRS-A Score')
ax.set_ylabel('Trait Anxiety')
plt.legend(['Trait Anxiety', 'State Anxiety'])
ax.set_title('ASRS-A Score vs. Anxiety')
plt.savefig('fig15.png')
plt.show()


### Build classifiers to identify a game-based biomarker for ADHD

In [ ]:
devdata.info()

In [ ]:
### Use logistic regression to predict ADHD target:
devdata.head()
y = devdata['Target'].values
X = devdata.drop(['Target', 'ID', 'user_id'], axis=1)

# Normalize data:
X = (X - X.mean()) / X.std()

# Split data into training and testing sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.reset_index(drop=True)
X_train
X_test = X_test.reset_index(drop=True)

# models = []
# models.append(("LR",LogisticRegression()))
# models.append(("NB",GaussianNB()))
# models.append(("RF",RandomForestClassifier()))
# models.append(("SVC",SVC()))
# models.append(("Dtree",DecisionTreeClassifier()))
# models.append(("KNN",KNeighborsClassifier()))

# for name,model in models:
#    kfold = KFold(n_splits=2)
#    cv_result = cross_val_score(model,X_train,y_train, cv = kfold,scoring = "accuracy")
#    print(cv_result)

# Transpose matrices:
# X_train = X_train.T
# X_test = X_test.T

# Fit logistic regression model:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
accuracy_score = logreg.score(X_test, y_test)
print('Accuracy: ', accuracy_score)

